# tools

> The hands: what the application under the agent must provide, and every tool built on top of it.

`Host` is the whole dependency this package has on the world. Everything else here is
derived from it -- the tools the model is given, the skills it can read, the extensions a
user can add, and the sub-agents it can delegate to. A host that satisfies the signatures
but not the contracts is a host that quietly hands an agent the whole filesystem, so the
docstrings in `Host` are the specification.

In [ ]:
#| default_exp tools

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail, expect_fail
from ramabana.testing import MemHost, FakeBackend

In [ ]:
#| export
import ast, functools, json, os, re, runpy, shutil, threading, uuid
from dataclasses import dataclass, field
from pathlib import Path
from fastcore.basics import AttrDict
from fastcore.docments import frontmatter
from ramabana.core import AgentError, agent_err

## The host

`Host` is the application under an agent, as an interface. Every method may raise, and
the tools catch rather than let an exception end a turn. A capability that cannot be
supported raises `NotImplementedError`, which `tools_for` reads as "do not offer this tool"
-- an agent told about a tool that always fails is worse off than one never told about it.

In [ ]:
#| export
class Hit:
    "One search result, in the shape every backend of `Host.search` returns."
    def __init__(self, path, line=1, symbol='', text=''):
        self.path, self.line, self.symbol, self.text = path, line, symbol, text
    def __repr__(self): return f'{self.path}:{self.line}  {self.symbol}  {self.text}'

In [ ]:
#| export
class Host:
    """The application under an agent.

    Every method may raise; the tools in `tools.py` catch and report rather than let an
    exception end a turn. Methods that cannot be supported should raise
    `NotImplementedError`, which the tool list reads as "do not offer this tool" -- an
    agent told about a tool that always fails is worse off than one never told about it.
    """

    # -- where it is allowed to be -------------------------------------------
    @property
    def roots(self):
        "The open folders, as absolute paths. The agent is told about these and confined to them."
        raise NotImplementedError

    def check(self, path, must_exist=False):
        """Resolve `path` and refuse anything outside `roots`, returning a `Path`.

        This is the single chokepoint. exhash and fossick both write to disk on their own
        account, so each is handed a path this has already approved rather than a path the
        model supplied. Every other method here may assume its argument came through here.
        """
        raise NotImplementedError

    def walk(self):
        "Every readable file under the open folders."
        raise NotImplementedError

    def read(self, path):
        "One file's text, or None when it cannot be read."
        raise NotImplementedError

    def write(self, path, text):
        "Write `text` to `path`, through the same sandbox `check` enforces. Returns the path written."
        raise NotImplementedError

    def text_at(self, path):
        """One file as a single diffable document, `''` when it does not exist yet, None on error.

        Distinct from `read` because a notebook diffs as cell sources rather than as
        nbformat JSON, and because a file the agent is about to create must diff as a pure
        addition instead of as an error. This is what `Agent.changes()` compares.
        """
        raise NotImplementedError

    # -- seeing the code -----------------------------------------------------
    def search(self, query, limit=20):
        "Search the code index for `query`, returning `Hit`s. Semantic if an index exists, literal if not."
        raise NotImplementedError

    def peers(self, path, line, limit=20):
        "Code shaped like whatever is defined at `path`:`line` -- every place a pattern was already used."
        raise NotImplementedError

    def symbols(self, path):
        "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."
        raise NotImplementedError

    @property
    def search_note(self):
        "Which engine answered, and anything it wants to say about why. Shown when a search finds nothing."
        return ''

    # -- reading the web -----------------------------------------------------
    def web_search(self, query, n=20):
        "Search the web; returns objects with `.title` and `.url`."
        raise NotImplementedError

    def read_url(self, url, remember=True):
        "One page as markdown; `remember=False` keeps sensitive/low-quality results ephemeral."
        raise NotImplementedError

    def research(self, query):
        "Search and read the top results into one cited digest. Slower than `web_search`."
        raise NotImplementedError

    @property
    def research_note(self): return ''

    # -- durable research memory --------------------------------------------
    def memory_search(self, query, limit=8):
        "Search remembered pages as whole tree sections, returning structured rows."
        raise NotImplementedError

    def memory_tree(self, document=''):
        "The heading tree for remembered documents; an empty document lists every root."
        raise NotImplementedError

    def memory_read(self, node_id):
        "Read one remembered section and its children by stable node id."
        raise NotImplementedError

    def memory_topics(self, limit=12):
        "Labelled semantic clusters across remembered research."
        raise NotImplementedError

    def memory_forget(self, doc_id):
        "Purge one remembered document and all derived tree/chunk/vector data."
        raise NotImplementedError

    # -- standing interests --------------------------------------------------
    # Memory above is what has already been read. This is what the agent has arranged to
    # read *later*. A watch is a job the host re-runs on an interval; `poll` is the tick a
    # scheduler, a cron or a frontend calls. A reminder is the degenerate case -- a watch
    # whose action is simply to file its own text back into memory when it comes due --
    # and it is here rather than in the frontend because the thing being reminded of is
    # usually the thing that was remembered, and both should live in one store.
    def remember(self, text, title=None, tags=()):
        "File `text` into durable memory as a note. Returns the document record."
        raise NotImplementedError

    def watch(self, target, action='remind', every='1d', note=None, **params):
        "Register a recurring job. `target` is a URL, a query, or the text of a reminder."
        raise NotImplementedError

    def watches(self, due_only=False):
        "Every registered watch, soonest first. `due_only` keeps the ones that have come due."
        raise NotImplementedError

    def unwatch(self, watch_id):
        "Delete one watch. Whatever it already filed stays in memory."
        raise NotImplementedError

    def poll(self):
        """Run every watch that is due and report what fired.

        One failing watch must not stop the rest: a host implementing this records the error
        on the row and carries on, because a dead URL should not silence a reminder.
        """
        raise NotImplementedError

    @property
    def watch_actions(self):
        "The `action` values this host's `watch` will accept."
        return ('remind',)

    # -- notebooks -----------------------------------------------------------
    # The harness deliberately does not own a notebook representation. exhash addresses
    # cells by path and id without one, so only the two operations that genuinely need to
    # know what a notebook *is* are delegated here.
    def nb_cells(self, path):
        "`[(id, cell_type, first_line)]` for one notebook."
        raise NotImplementedError

    def nb_add_cell(self, path, source, index=-1, cell_type='code'):
        "Insert a cell (-1 appends), creating the notebook if needed. Returns the new cell's id."
        raise NotImplementedError

    # -- the live session ----------------------------------------------------
    def run_python(self, code):
        """Run `code` in the user's live namespace under whatever restrictions the host imposes.

        The contract the agent is briefed on: read anything, bind results to new names,
        never rebind or delete the owner's. Enforcing it is the host's job -- the harness
        only promises to tell the model about it.
        """
        raise NotImplementedError

    def inspect_python(self, code, scope='isolated'):
        """Run `code` against the live namespace without touching what the user has.

        Two scopes, and the difference is the interpreter you get rather than the safety you
        get -- both protect the owner's variables, by different means:

        - `'isolated'` runs in an allowlist sandbox on a *copy*. Attribute reads and builtins
          work; most library method calls are refused. Nothing can reach the real namespace
          at all, which is why it is the default and why it needs no trust.
        - `'overlay'` runs the real interpreter against the real namespace under an AST
          policy: the agent may read anything and bind its own names, which persist in its
          own layer, and cannot delete, rebind in place, or mutate the owner's. `list(df.columns)`
          and `df.head().to_dict()` work here; in the sandbox they do not.

        A host may refuse `'overlay'` (see `scopes`) and fall back to isolated, which is what
        a locked-down deployment does. Under a concurrent kernel either scope runs *alongside*
        a busy cell rather than queueing behind it.
        """
        raise NotImplementedError

    @property
    def scopes(self):
        "The scopes `inspect_python` will actually honour, most trusted last."
        return ('isolated',)

    @property
    def kernel_kind(self):
        """What runs the live namespace, and whether it can execute concurrently.

        `'ipymini'` means an inspection can run while a cell is busy; anything else means
        it queues behind whatever the kernel is already doing. The agent is told which,
        because "read the dataframe" is good advice under one and a way to hang the session
        under the other.
        """
        return 'ipykernel'

    @property
    def concurrent(self): return self.kernel_kind == 'ipymini'

    def list_vars(self):
        "What is in the live namespace: name, type, and a short value, one per line."
        raise NotImplementedError

    def terminal_text(self, lines=200):
        "What the IDE's terminal has printed. Read-only: this shows what the user ran, it cannot run anything."
        raise NotImplementedError

    # -- the person ----------------------------------------------------------
    @property
    def approvals(self):
        "The `Approvals` this host uses to put a write in front of a person, or None to approve everything."
        return None

    def note(self, text):
        "Tell the user something out of band (a status line). Never blocks; a host may drop it."
        pass

A `Hit` is the one shape every search backend returns, whether the index behind it is
semantic or a literal scan.

In [ ]:
Hit('nbs/02_tools.ipynb', 42, 'tools_for', 'def tools_for(host, get_skills=None, extra=()):')

`NullHost` is a host with nothing behind it, and it is the reference implementation of
"absent" -- the harness runs bare on it, which is how the probing in `tools_for` gets
tested without a real application.

In [ ]:
#| export
class NullHost(Host):
    "A host with nothing behind it: every capability absent, so the harness runs bare in a test."

    def __init__(self, roots=()): self._roots = [str(r) for r in roots]

    @property
    def roots(self): return self._roots

    def check(self, path, must_exist=False):
        from pathlib import Path
        return Path(path)

    def walk(self): return []
    def read(self, path): return None
    def write(self, path, text): raise NotImplementedError
    def text_at(self, path): return None
    def search(self, query, limit=20): return []
    def peers(self, path, line, limit=20): return []
    def symbols(self, path): return []
    def web_search(self, query, n=20): return []
    def read_url(self, url, remember=True): return None
    def research(self, query): return ''
    def memory_search(self, query, limit=8): raise NotImplementedError
    def memory_tree(self, document=''): raise NotImplementedError
    def memory_read(self, node_id): raise NotImplementedError
    def memory_topics(self, limit=12): raise NotImplementedError
    def memory_forget(self, doc_id): raise NotImplementedError
    def remember(self, text, title=None, tags=()): raise NotImplementedError
    def watch(self, target, action='remind', every='1d', note=None, **params): raise NotImplementedError
    def watches(self, due_only=False): raise NotImplementedError
    def unwatch(self, watch_id): raise NotImplementedError
    def poll(self): raise NotImplementedError
    def nb_cells(self, path): raise NotImplementedError
    def nb_add_cell(self, path, source, index=-1, cell_type='code'): raise NotImplementedError
    def run_python(self, code): raise NotImplementedError
    def inspect_python(self, code, scope='isolated'): raise NotImplementedError
    def list_vars(self): raise NotImplementedError
    def terminal_text(self, lines=200): raise NotImplementedError

In [ ]:
h = NullHost(['/proj'])
h.roots, h.walk(), h.read('/proj/a.py')

Two properties are answered rather than raised, because the agent is briefed on them
before it calls anything: which inspection scopes are honoured, and whether the kernel can
run an inspection while a cell is busy. "Read the dataframe" is good advice under one and a
way to hang the session under the other.

In [ ]:
h.scopes, h.kernel_kind, h.concurrent

In [ ]:
with expect_fail(NotImplementedError): h.run_python('1+1')
h.approvals is None

## A host over real folders

`LocalHost` is the reference implementation: enough of a host to run the agent from a
terminal, from an MCP server or from a test, with no IDE anywhere. It is also where the
sandbox actually lives -- `check` resolves `..` and symlinks *before* comparing against the
open folders, so every other method may assume its argument was approved.

In [ ]:
#| export
SANDBOX = 'path is outside the open folders'
SKIP_DIRS = frozenset({'.git', '.hg', '.svn', '__pycache__', '.venv', 'venv', 'node_modules',
                       '.ipynb_checkpoints', '.pytest_cache', '.mypy_cache', '_docs', '_proc',
                       'dist', 'build', '.quarto', '.idea', '.attic'})
SKIP_SUFFIXES = frozenset({'.pyc', '.pyo', '.so', '.dylib', '.dll', '.a', '.o', '.zip', '.gz',
                           '.whl', '.png', '.jpg', '.jpeg', '.gif', '.webp', '.pdf', '.parquet',
                           '.sqlite', '.db', '.bin', '.safetensors', '.gguf'})
MAX_FILE = 2_000_000      # bytes; a file larger than this is data, not source
MAX_VARS = 200
LD_CHARS = 4000           # of a page's JSON-LD to keep; enough for a product, not a catalogue

_LD = re.compile(r'<script[^>]+application/ld\+json[^>]*>(.*?)</script>', re.S | re.I)

def ld_json(html):
    "The `schema.org` JSON-LD blocks in `html` -- where a page states its price, author or rating."
    out = []
    for m in _LD.finditer(html or ''):
        try: out.append(json.loads(m.group(1)))
        except Exception: pass
    return out

class LocalHost(Host):
    """A host over real folders on disk, and a live Python namespace in this process.

    This is the reference implementation of `Host`: enough of one to run the agent from a
    terminal, from an MCP server, or from a test, with no IDE anywhere. Capabilities it
    genuinely cannot provide raise `NotImplementedError`, so `tools_for` drops them rather
    than offering the model a tool that always fails.
    """

    def __init__(self,
                 roots=('.',),          # the folders the agent is confined to
                 ns=None,               # the live namespace; a fresh dict when None
                 approvals=None,        # an `Approvals`, or None to gate nothing
                 note=None,             # callable for out-of-band status lines
                 web=True,              # wire the web tools to fossick when it is installed
                 index=True):           # start a Kosha sync for every open root
        self._roots = [str(Path(r).expanduser().resolve()) for r in roots]
        self.ns = {'__name__': '__main__'} if ns is None else ns
        self._approvals, self._note, self.web = approvals, note, web
        self.transcript = []           # what this process has printed, for `read_terminal`
        self._koshas, self._index_errors, self._index_thread = [], [], None
        if index: self.sync_index()

    def sync_index(self, wait=False, force=False):
        """Run `Kosha.sync` for every open root, once, in a daemon thread.

        Indexing starts when the host does, rather than on the first search: by then the
        model is already waiting. Kosha's sync is incremental, so an existing `.kosha`
        usually becomes ready immediately; the first run parses, embeds and graphs the repo.
        `wait=True` is for a command or test that needs semantic results now.
        """
        if self._index_thread is None or not self._index_thread.is_alive():
            def run():
                try:
                    # Kosha uses tqdm internally even with `verbose=False`; suppress it before
                    # import so a background sync never writes through teleprint's live tail.
                    os.environ.setdefault('TQDM_DISABLE', '1')
                    from kosha import Kosha
                    self._koshas = [Kosha(dir=Path(root)) for root in self._roots]
                    for k, root in zip(self._koshas, self._roots):
                        # `sync`, not a private subset: code store, environment metadata and
                        # graph stay mutually consistent. It is incremental after first use.
                        k.sync(dir=Path(root), verbose=False, force=force, pyproject=True,
                               in_parallel=False, sync_graph=force)
                except Exception as e:
                    self._index_errors.append(agent_err(e))
            self._index_thread = threading.Thread(target=run, name='ramabana-kosha-sync', daemon=True)
            self._index_thread.start()
        if wait: self._index_thread.join()
        return self

    @property
    def index_ready(self):
        return bool(self._koshas) and self._index_thread is not None and not self._index_thread.is_alive()

    def wait_index(self, timeout=None):
        "Wait for the automatic Kosha sync. Returns whether semantic search is ready."
        if self._index_thread is not None: self._index_thread.join(timeout)
        return self.index_ready

    @property
    def roots(self): return list(self._roots)

    def check(self, path, must_exist=False):
        "Resolve `path` and refuse anything outside `roots`. Every other method assumes this ran."
        p = Path(path).expanduser()
        if not p.is_absolute(): p = Path(self._roots[0])/p
        # `resolve` before comparing, so `a/../../etc/passwd` and a symlink out both fail here.
        p = p.resolve()
        if not any(p == Path(r) or Path(r) in p.parents for r in self._roots):
            raise AgentError(f'{SANDBOX}: {p}')
        if must_exist and not p.exists(): raise AgentError(f'no such file: {p}')
        return p

    def _walk(self, root):
        for p in sorted(Path(root).rglob('*')):
            if any(part in SKIP_DIRS for part in p.parts): continue
            if not p.is_file() or p.is_symlink(): continue
            if p.suffix.lower() in SKIP_SUFFIXES: continue
            try:
                if p.stat().st_size > MAX_FILE: continue
            except OSError: continue
            yield p

    def walk(self):
        return [p for r in self._roots for p in self._walk(r)]

    def read(self, path):
        try: return self.check(path, must_exist=True).read_text(encoding='utf-8')
        except Exception: return None

    def write(self, path, text):
        p = self.check(path)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(str(text), encoding='utf-8')
        return str(p)

    def text_at(self, path):
        """One file as a diffable document: a notebook as its cell sources, anything else as text.

        `''` rather than None for a file that does not exist yet, so a file the agent is
        about to create diffs as a pure addition instead of as an error.
        """
        try: p = self.check(path)
        except Exception: return None
        if not p.exists(): return ''
        if p.suffix == '.ipynb':
            try:
                from fastcore.nbio import read_nb
                return '\n\n'.join(''.join(c.source) for c in read_nb(p).cells)
            except Exception: return None
        try: return p.read_text(encoding='utf-8')
        except Exception: return None

    # -- seeing the code -----------------------------------------------------
    def _rg(self, query, limit):
        "Literal search through ripgrep, when it is installed: it is an order of magnitude faster."
        import subprocess
        if not shutil.which('rg'): return None
        cmd = ['rg', '--line-number', '--no-heading', '--fixed-strings', '--max-count', '5',
               '--max-filesize', str(MAX_FILE), '-e', query, *self._roots]
        try: out = subprocess.run(cmd, capture_output=True, text=True, timeout=20).stdout
        except Exception: return None
        hits = []
        for line in out.splitlines()[:limit]:
            path, _, rest = line.partition(':')
            num, _, text = rest.partition(':')
            if not num.isdigit(): continue
            hits.append(Hit(path, int(num), '', text.strip()[:200]))
        return hits

    def _semantic(self, query, limit):
        "Kosha's repo-first hybrid results as the Host's stable `Hit` shape."
        if not self.index_ready: return []
        out, seen = [], set()

        def add(rows):
            for row in rows:
                row = dict(row)
                meta = row.get('metadata') or {}
                if isinstance(meta, str):
                    try: meta = ast.literal_eval(meta)
                    except Exception: meta = {}
                path = str(meta.get('path') or row.get('path') or '')
                line = int(meta.get('lineno') or 1)
                key = (path, line)
                if key in seen: continue
                seen.add(key)
                symbol = meta.get('mod_name') or meta.get('name') or ''
                text = ' '.join(str(row.get('content') or '').split())[:240]
                out.append(Hit(path, line, str(symbol), text))
                if len(out) >= limit: return True
            return False

        # The open repository comes first. Fill any room from Kosha's environment index,
        # which is how `search_code` can find an installed library's implementation too.
        for k in self._koshas:
            try:
                if add(k.repo_context(query, columns='content,path,metadata')): return out
            except Exception as e: self._index_errors.append(agent_err(e))
        for k in self._koshas:
            try:
                if add(k.context(query, limit=limit, repo=False, env=True, graph=True,
                                 columns='content,metadata')): return out
            except Exception as e: self._index_errors.append(agent_err(e))
        return out

    def search(self, query, limit=20):
        "Semantic + keyword search through Kosha, with a literal fallback while its first sync runs."
        if not (query or '').strip(): return []
        if (hits := self._semantic(query, limit)): return hits
        if (hits := self._rg(query, limit)) is not None: return hits
        hits = []
        for p in self.walk():
            try: text = p.read_text(encoding='utf-8')
            except Exception: continue
            if query not in text: continue
            for i, line in enumerate(text.splitlines(), 1):
                if query in line:
                    hits.append(Hit(str(p), i, '', line.strip()[:200]))
                    if len(hits) >= limit: return hits
        return hits

    @property
    def search_note(self):
        if self.index_ready: return f'Kosha semantic + keyword index over {len(self._roots)} folder(s) and environment'
        if self._index_errors: return f'Kosha unavailable ({self._index_errors[-1]}); literal fallback'
        return 'Kosha sync in progress; literal fallback' + (' via ripgrep' if shutil.which('rg') else '')

    def _defs(self, path):
        "Every def/class in one file as `(line, qualified_name, depth)`, by parsing rather than grepping."
        src = self.read(path)
        if src is None: return []
        try: tree = ast.parse(src)
        except SyntaxError: return []
        out = []
        def walk(node, prefix='', depth=0):
            for child in ast.iter_child_nodes(node):
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                    name = f'{prefix}{child.name}'
                    out.append((child.lineno, name, depth))
                    walk(child, f'{name}.', depth + 1)
        walk(tree)
        return out

    def symbols(self, path):
        "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."
        p = self.check(path)
        out = []
        for line, name, depth in self._defs(p):
            h = Hit(str(p), line, name, '')
            h.score = depth
            out.append(h)
        return out

    def peers(self, path, line, limit=20):
        """Every other place the symbol defined at `path`:`line` is mentioned.

        Not a semantic index -- this host has none -- but the useful half of one: the call
        sites and overrides of the thing under the cursor, which is what "where else do we
        do this" usually means.
        """
        p = self.check(path)
        defs = self._defs(p)
        name = next((n for ln, n, _ in sorted(defs, key=lambda d: -d[0]) if ln <= int(line)), None)
        if name is None: return []
        leaf = name.split('.')[-1]
        return [h for h in self.search(leaf, limit * 2)
                if not (str(h.path) == str(p) and h.line == int(line))][:limit]

    # -- notebooks -----------------------------------------------------------
    def nb_cells(self, path):
        from fastcore.nbio import read_nb
        nb = read_nb(self.check(path, must_exist=True))
        return [(c.get('id', ''), c.cell_type, ''.join(c.source)) for c in nb.cells]

    def nb_add_cell(self, path, source, index=-1, cell_type='code'):
        from fastcore.nbio import read_nb, write_nb, mk_cell, dict2nb
        p = self.check(path)
        nb = read_nb(p) if p.exists() else dict2nb({'cells': [], 'metadata': {}, 'nbformat': 4, 'nbformat_minor': 5})
        cell = mk_cell(source, cell_type)
        if not cell.get('id'): cell['id'] = uuid.uuid4().hex[:8]
        nb.cells.append(cell) if index < 0 else nb.cells.insert(int(index), cell)
        p.parent.mkdir(parents=True, exist_ok=True)
        write_nb(nb, p)
        return cell['id']

    # -- the live session ----------------------------------------------------
    def _exec(self, code, ns):
        """Run `code` in `ns`, returning printed output plus the last expression's value.

        Split into statements-then-final-expression so `df.shape` answers with the shape
        instead of with nothing, which is what makes this usable for looking at state.
        """
        import contextlib, io
        buf = io.StringIO()
        tree = ast.parse(str(code))
        last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            if tree.body: exec(compile(tree, '<agent>', 'exec'), ns)
            value = eval(compile(ast.Expression(last.value), '<agent>', 'eval'), ns) if last else None
        out = buf.getvalue()
        if value is not None: out += ('' if not out or out.endswith('\n') else '\n') + repr(value)
        return out.strip() or '(no output)'

    def run_python(self, code):
        "Run `code` in the live namespace. Failures come back as text: a tool cannot usefully raise."
        try: return self._exec(code, self.ns)
        except Exception as e: return f'{agent_err(e)}'

    def inspect_python(self, code, scope='isolated'):
        "Run `code` against a *copy* of the namespace, so nothing the user made can move."
        if scope not in self.scopes: return f'this host only honours {self.scopes}'
        try: return self._exec(code, dict(self.ns))
        except Exception as e: return f'{agent_err(e)}'

    @property
    def scopes(self):
        "Isolated only. Overlay needs an AST policy over the real namespace, which belongs to an IDE."
        return ('isolated',)

    @property
    def kernel_kind(self): return 'inprocess'

    def list_vars(self):
        rows = []
        for k, v in list(self.ns.items())[:MAX_VARS]:
            if k.startswith('_') or callable(v) or isinstance(v, type(ast)): continue
            try: short = repr(v)
            except Exception: short = '<unreprable>'
            rows.append(f'{k:20} {type(v).__name__:12} {short[:60]}')
        return '\n'.join(rows)

    def terminal_text(self, lines=200):
        "What this process has printed, when the application records it in `transcript`."
        return '\n'.join(str(x) for x in self.transcript[-int(lines):])

    # -- the web -------------------------------------------------------------
    def _fossick(self):
        if not self.web: raise NotImplementedError
        try:
            import fossick
            return fossick
        except Exception: raise NotImplementedError

    def web_search(self, query, n=20):
        """Search the web through fossick.

        An empty query answers `[]` after checking only that fossick imports, because that is
        how `tools_for` probes for this capability -- and a tool list that cannot be built
        without a network round trip is a tool list that fails on a train.
        """
        fossick = self._fossick()
        if not str(query).strip(): return []
        rows = fossick.search(str(query))[:int(n)]
        return [AttrDict(title=str(r.get('title', '')), url=str(r.get('href') or r.get('url', ''))) for r in rows]

    #: Extracted characters below which a page did not really load. A site that turns away
    #: scrapers does not answer 403 -- it answers 200 with an empty shell, so escalating on
    #: the status code never fires and fossick's own `auto=` tier stops at `plain`.
    THIN_PAGE = 400

    def read_url(self, url, remember=True):
        """One page as markdown: the prose, and the structured data the prose leaves out.

        Two things go wrong on a modern page and neither shows up as an error. It renders in
        the browser, so a plain fetch returns a shell -- answered by escalating to a real
        browser when the extracted text comes back too thin to be a page. And its *facts*
        live in `schema.org` JSON-LD rather than in its prose, so readability extraction on a
        product page faithfully keeps the ingredient list and throws the price away. Both are
        general: the JSON-LD block is a standard, not a selector for one shop.
        """
        fossick = self._fossick()
        page = fossick.fetch(str(url))
        text = str(fossick.to_md(page) or '')
        if len(text.strip()) < self.THIN_PAGE:
            # Only now: a stealth browser costs ten seconds and a Chrome, and most pages
            # never need one.
            try:
                heavy = fossick.fetch(str(url), stealthy=True)
                page, text = heavy, str(fossick.to_md(heavy) or '') or text
            except Exception: pass
        if (ld := ld_json(getattr(page, 'html_content', '') or '')):
            text = f'<structured-data>\n{json.dumps(ld)[:LD_CHARS]}\n</structured-data>\n\n{text}'
        return None if not text.strip() else AttrDict(text=text, url=str(url))

    def research(self, query):
        return str(self._fossick().research(str(query)) or '')

    @property
    def research_note(self): return 'fossick' if self.web else 'web access is switched off'

    # -- the person ----------------------------------------------------------
    @property
    def approvals(self): return self._approvals

    def note(self, text):
        self.transcript.append(str(text))
        if self._note:
            try: self._note(str(text))
            except Exception: pass

A root with a little source in it:

In [ ]:
root = Path(tempfile.mkdtemp()).resolve()/'proj'
(root/'pkg').mkdir(parents=True)
(root/'pkg'/'sizes.py').write_text(
    'RESERVE = 16_384\n\n'
    'def threshold(ctx, reserve=RESERVE):\n'
    '    "Where compaction becomes due."\n'
    '    if not ctx: return None\n'
    '    return max(1, ctx - min(reserve, ctx // 4))\n\n'
    'class Budget:\n'
    '    def spend(self, n): return threshold(n)\n')
(root/'pkg'/'use.py').write_text('from .sizes import threshold\n\nprint(threshold(4096))\n')
local = LocalHost([root])
[str(p.relative_to(root)) for p in local.walk()]

The sandbox is the whole reason this class exists. A path that climbs out is refused before
anything reads it, and so is a symlink that points out -- which is why `check` resolves
first and compares second.

In [ ]:
test_fail(lambda: local.check('../../etc/passwd'), contains='outside the open folders')
(root/'escape').symlink_to('/etc')
test_fail(lambda: local.check('escape/passwd'), contains='outside the open folders')
local.check('pkg/sizes.py')

A relative path is taken against the first open folder, and a missing file can be demanded
here rather than discovered three calls later.

In [ ]:
test_fail(lambda: local.check('pkg/nope.py', must_exist=True), contains='no such file')
local.read('pkg/sizes.py').splitlines()[0]

Construction starts `Kosha.sync` in a daemon thread for every open root. It starts before the
model does so indexing overlaps model startup; an existing `.kosha` is incremental and is
normally ready before the first search.

In [ ]:
local.wait_index(120), local.search_note

Kosha combines keyword and semantic retrieval, with the open repository ahead of its index
of installed packages. This query does not contain `threshold`, but the function's meaning
is enough to retrieve it.

In [ ]:
hits = local.search('where compaction becomes due')
[(h.path, h.line, h.symbol) for h in hits[:3]]

In [ ]:
test_eq(hits[0].symbol.endswith('threshold'), True)
test_eq(Path(hits[0].path), root/'pkg'/'sizes.py')
local.index_ready

While the first sync is still running, search falls back to literal ripgrep rather than
making the model wait. `search_note` always says which engine answered, so "no matches" can
be told apart from "the semantic index was not ready".

In [ ]:
test_eq(local.web_search(''), [])          # the capability probe must not hit the network
local.research_note

In [ ]:
local.search('threshold')[:2], local.search_note

Symbols come from parsing rather than grepping, so a method is reported at its own depth and
a `def` inside a docstring is not reported at all.

In [ ]:
[(h.symbol, h.line, h.score) for h in local.symbols('pkg/sizes.py')]

`peers` answers "where else do we do this" with the mentions of whatever is defined at that
line. Not a semantic index -- this host has none -- but the useful half of one.

In [ ]:
local.peers('pkg/sizes.py', 3)

`text_at` is what `Agent.changes()` diffs, and it is notebook-aware: a notebook diffs as its
cell sources rather than as nbformat JSON. A file that does not exist yet is `''`, so
creating one diffs as a pure addition instead of as an error.

In [ ]:
cid = local.nb_add_cell('nb/demo.ipynb', 'x = threshold(4096)\nx')
local.nb_cells('nb/demo.ipynb'), local.text_at('nb/demo.ipynb')

In [ ]:
test_eq(local.text_at('pkg/nope.py'), '')
cid

The live namespace is real, and it persists between calls -- which is what makes the agent's
"bind results to new names" contract mean anything.

In [ ]:
local.run_python('import math\nradii = [1, 2, 3]'), local.run_python('areas = [math.pi*r*r for r in radii]\nareas[:2]')

A trailing expression is answered with its value rather than with silence, since that is what
makes the tool usable for looking at state at all.

In [ ]:
local.run_python('len(areas)'), local.run_python('print("a side effect")')

Failures come back as text. A tool that raises ends a turn, and a misspelled name is not
worth ending a turn over.

In [ ]:
local.run_python('no_such_name + 1')

`inspect_python` runs against a *copy*, so nothing done there can move what the user made.
This host advertises only the isolated scope: an overlay needs an AST policy over the real
namespace, which is an IDE's job.

In [ ]:
local.inspect_python('radii.append(99)\nlen(radii)'), local.run_python('len(radii)')

In [ ]:
test_eq(local.scopes, ('isolated',))
print(local.list_vars())

## Skills

A skill is know-how the agent can read on demand: a package that documents itself, or a
`SKILL.md` file in a repository. Only names and one-line descriptions go in the system
prompt; the bodies are fetched by the `read_skill` tool, which is what keeps a dozen skills
affordable.

In [ ]:
#| export
GROUP,EXTRA_MODULES,MAX_SKILL_CHARS = 'pyskills',('exhash.skill',),20_000

def _describe(text, mx=300):
    'A one-line description from a skill body: its first paragraph, collapsed.'
    body = (text or '').strip()
    if not body: return ''
    para = body.split('\n\n', 1)[0]
    one = ' '.join(para.split())
    return one if len(one) <= mx else one[:mx - 1].rstrip() + '…'

@dataclass
class Skill:
    'One skill: how to name it, when it applies, and how to get the whole text.'
    name: str
    source: str                   # 'pyskill' | 'md'
    description: str = ''
    where: str = ''               # module path or file path, shown so a person can go read it
    _text: object = field(default=None, repr=False)

    def text(self):
        'The full skill body, clipped. Never raises: a broken skill reports itself as one.'
        try: t = self._text() if callable(self._text) else (self._text or '')
        except Exception as e: return f'could not read skill {self.name}: {agent_err(e)}'
        t = str(t)
        return t if len(t) <= MAX_SKILL_CHARS else t[:MAX_SKILL_CHARS] + f'\n…[{len(t)-MAX_SKILL_CHARS} more chars]'

    def dict(self): return {'name': self.name, 'source': self.source,'description': self.description, 'where': self.where}

A description is the first paragraph, collapsed to one line -- the same convention a
`SKILL.md` frontmatter `description` follows, so a package and a file read alike in the
index.

In [ ]:
_describe('''Search the web and read results into durable memory.

The rest of this document explains the memory layout, which the index does not need.''')

A `Skill` holds a *way to get* the body rather than the body, so discovering forty skills
does not read forty files. The body is clipped, and a loader that raises reports itself as
the skill's text instead of taking the turn down with it.

In [ ]:
s = Skill('nbdev', 'md', 'Develop nbdev projects.', 'nbs/SKILL.md', _text=lambda: 'the whole skill body')
s.text(), s.dict()

In [ ]:
def _explodes(): raise FileNotFoundError('SKILL.md')
Skill('broken', 'md', _text=_explodes).text()

Skills come from installed packages (the `pyskills` entry-point group) and from
`<name>/SKILL.md` directories. The precedence is deliberate: a file beats a package,
because the package's skill is the general advice and the one in your own repository is
the correction.

In [ ]:
#| export
def _mod_skill(name, modpath):
    "A `Skill` for a module, without importing it until someone asks for the body."
    def load():
        from importlib import import_module
        return import_module(modpath).__doc__ or ''
    # The description does need the docstring, and there is no way to read one without
    # importing. Failing quietly is right: a package whose import breaks should cost the
    # agent one missing skill, not a session.
    try:
        from importlib import import_module
        doc = import_module(modpath).__doc__ or ''
    except Exception:
        return None
    if not doc.strip(): return None
    return Skill(name=name, source='pyskill', description=_describe(doc), where=modpath, _text=load)

def _pyskills():
    "Every module published under the `pyskills` entry-point group, plus the known stragglers."
    out, seen = [], set()
    try:
        from importlib.metadata import entry_points
        eps = list(entry_points(group=GROUP))
    except Exception:
        eps = []
    for ep in eps:
        mod = getattr(ep, 'value', None) or ep.name
        if mod in seen: continue
        seen.add(mod)
        if (s := _mod_skill(ep.name.split('.')[-1] or ep.name, mod)): out.append(s)
    for mod in EXTRA_MODULES:
        if mod in seen: continue
        seen.add(mod)
        if (s := _mod_skill(mod.split('.')[0], mod)): out.append(s)
    return out

def skill_dirs(roots=(), cfg=None):
    """Where SKILL.md files are looked for, in increasing precedence.

    User directories first so a project can override a personal skill of the same name --
    which is the way round that matters, since the project is the shared thing and the
    personal one is the habit.
    """
    from pathlib import Path
    ds = []
    if cfg is not None: ds.append(Path(cfg)/'skills')
    ds.append(Path.home()/'.agents'/'skills')
    for r in roots: ds += [Path(r)/'.leela'/'skills', Path(r)/'.agents'/'skills']
    return ds

def _md_skills(d):
    "Skills in one directory, following the Agent Skills layout: `<name>/SKILL.md`."
    from pathlib import Path
    d = Path(d)
    if not d.is_dir(): return []
    out = []
    for p in sorted(d.iterdir()):
        if not p.is_dir(): continue
        f = p/'SKILL.md'
        if not f.exists(): continue
        try: raw = f.read_text(encoding='utf-8')
        except Exception: continue
        meta, body = frontmatter(raw)
        out.append(Skill(name=meta.get('name') or p.name, source='md',
                         description=meta.get('description') or _describe(body),
                         where=str(f), _text=body))
    return out

def discover(roots=(), cfg=None, extra=()):
    """Every skill available to this agent, later sources winning on a name clash.

    Order is pyskills, then each skill directory in `skill_dirs` order, then `extra` (what
    an extension registered). A file beats a package deliberately: the package's skill is
    the general advice, and the one you wrote in your own repository is the correction.
    """
    by_name = {}
    for s in _pyskills(): by_name[s.name] = s
    for d in skill_dirs(roots, cfg):
        for s in _md_skills(d): by_name[s.name] = s
    for s in extra or (): by_name[s.name] = s
    return sorted(by_name.values(), key=lambda s: s.name)

The search path, in increasing precedence -- personal habits first, so a project can
override them:

In [ ]:
[str(p) for p in skill_dirs(roots=['/proj'], cfg='/home/k/.config/leela')]

Written out, a skill directory follows the Agent Skills layout, and its frontmatter
supplies the name and description:

In [ ]:
tmp = Path(tempfile.mkdtemp())
d = tmp/'.agents'/'skills'/'notebook-tests'
d.mkdir(parents=True)
(d/'SKILL.md').write_text('---\nname: notebook-tests\ndescription: Write tests as notebook cells.\n---\n\nUse `test_eq`.\n')
[(x.name, x.source, x.description) for x in _md_skills(tmp/'.agents'/'skills')]

`discover` merges every source into one list, with later sources winning on a name clash.

In [ ]:
skills = discover(roots=[tmp])
[x.name for x in skills]

`skill_index` is what goes in the system prompt, and `find` is how `read_skill` resolves
what the model asked for.

In [ ]:
#| export
def skill_index(skills):
    "The block that goes in the system prompt: names and descriptions, never bodies."
    if not skills: return ''
    rows = '\n'.join(f'- `{s.name}` — {s.description}' for s in skills)
    return ('\n\n## Skills\n\nKnow-how available to you. Read one with `read_skill(name)` when its '
            'description matches what you are about to do, *before* you do it — several of these '
            'describe tools already installed in this environment, so the code they discuss is '
            'also searchable with `search_code`.\n\n' + rows)

def find(skills, name):
    """A skill by exact name, then by unique prefix, then by unique substring.

    Ambiguity returns None rather than a guess. A model that asked for `edit` and silently
    got `editskill` will read the wrong reference and then confidently do the wrong thing,
    which is worse than being told to be specific.
    """
    if not name: return None
    n = name.strip().lower()
    if (exact := [s for s in skills if s.name.lower() == n]): return exact[0]
    for pred in (lambda s: s.name.lower().startswith(n), lambda s: n in s.name.lower()):
        if len(hits := [s for s in skills if pred(s)]) == 1: return hits[0]
    return None

In [ ]:
print(skill_index([s for s in skills if s.name == 'notebook-tests']))

A name resolves exactly, then by unique prefix, then by unique substring. Ambiguity
returns `None` rather than a guess: a model that asked for `note` and silently got the
wrong skill will read the wrong reference and then confidently do the wrong thing.

In [ ]:
find(skills, 'notebook-tests'), find(skills, 'notebook'), find(skills, 'nope')

In [ ]:
two = [Skill('edit', 'md'), Skill('editor', 'md')]
test_eq(find(two, 'edit').name, 'edit')        # exact wins over prefix
find(two, 'edi') is None                        # ambiguous: say so

## Extensions

An extension is a Python file the user drops in a directory. `Registry` is everything it
is handed: tools, skills, slash commands, lifecycle hooks and the approval policy. What is
deliberately absent is any route to a backend's internals -- an extension that pokes at a
litert conversation would break on the next model switch, and break silently.

In [ ]:
#| export
EVENTS = ('before_turn', 'after_turn', 'before_tool', 'after_tool', 'compact', 'approval')

In [ ]:
#| export
class Registry:
    """What `setup(ext)` is handed: everything an extension may add, and nothing else.

    `host` and `agent` are exposed because an extension that cannot read a file or see the
    conversation is not worth writing. What is deliberately *not* here is any way to reach
    a backend's internals -- an extension that pokes at a litert conversation would break
    on the next model switch, and would break silently.
    """

    def __init__(self, host=None, agent=None):
        self.host, self.agent = host, agent
        self.tools, self.skills, self.commands = [], [], {}
        self.hooks = {e: [] for e in EVENTS}
        self.approve = None
        self.notes = []          # one line per extension: loaded, or why not

    # -- registration --------------------------------------------------------
    def tool(self, f):
        """Add a tool. Usable as a decorator.

        The contract is the backends' own: a plain function with type hints and a
        docstring. That docstring is what the model reads, so it is documentation and not
        a comment.
        """
        self.tools.append(f)
        return f

    def skill(self, name, text, description=''):
        "Add a skill the discovery pass would not find -- a file, a string, anything callable."
        s = Skill(name=name, source='ext', description=description or _describe(text if isinstance(text, str) else ''),
                  where='extension', _text=text)
        self.skills.append(s)
        return s

    def command(self, name, fn, help=''):
        "Add a slash command. `fn(agent, arg)` returns text for the frontend to show."
        self.commands[name.lstrip('/')] = (fn, help)
        return fn

    def on(self, event, fn):
        "Hook a harness lifecycle event. Unknown event names are an error, not a silent no-op."
        if event not in EVENTS: raise KeyError(f'unknown event {event!r}; known: {", ".join(EVENTS)}')
        self.hooks[event].append(fn)
        return fn

    def approval(self, fn):
        "Replace the approval policy wholesale. The last extension to call this wins."
        self.approve = fn
        return fn

    # -- dispatch ------------------------------------------------------------
    def fire(self, event, *args, **kw):
        "Run every hook for `event`, swallowing failures. Returns how many ran cleanly."
        n = 0
        for f in self.hooks.get(event, ()):
            try: f(*args, **kw); n += 1
            except Exception as e: self.notes.append(f'{event} hook failed: {agent_err(e)}')
        return n

In [ ]:
#| export
def ext_dirs(roots=(), cfg=None, project=False):
    "Where extensions are looked for. Project directories only when explicitly allowed."
    ds = []
    if cfg is not None: ds.append(Path(cfg)/'extensions')
    if project:
        for r in roots: ds.append(Path(r)/'.leela'/'extensions')
    return ds


def load(reg, roots=(), cfg=None, project=False, paths=()):
    """Run every extension found, calling its `setup(reg)`. Returns the registry.

    A file with no `setup` is loaded and left alone rather than reported as broken: that is
    how a shared helper module sitting in the same directory should behave.
    """
    files = []
    for d in ext_dirs(roots, cfg, project):
        if Path(d).is_dir(): files += sorted(p for p in Path(d).glob('*.py') if not p.name.startswith('_'))
    for p in paths or ():
        p = Path(p)
        files += sorted(p.glob('*.py')) if p.is_dir() else [p]
    for f in files:
        try:
            ns = runpy.run_path(str(f))
        except Exception as e:
            reg.notes.append(f'{f.name}: failed to load ({agent_err(e)})')
            continue
        fn = ns.get('setup')
        if not callable(fn):
            reg.notes.append(f'{f.name}: loaded, no setup()')
            continue
        before = (len(reg.tools), len(reg.skills), len(reg.commands))
        try: fn(reg)
        except Exception as e:
            reg.notes.append(f'{f.name}: setup() failed ({agent_err(e)})')
            continue
        d = [n - b for n, b in zip((len(reg.tools), len(reg.skills), len(reg.commands)), before)]
        reg.notes.append(f'{f.name}: {d[0]} tool(s), {d[1]} skill(s), {d[2]} command(s)')
    return reg

An extension is a file with a `setup(reg)`. This one adds a tool and a command:

In [ ]:
extdir = tmp/'extensions'
extdir.mkdir()
(extdir/'wordcount.py').write_text('''
def setup(reg):
    @reg.tool
    def word_count(path: str) -> str:
        "Count the words in a file in the open folders."
        return str(len((reg.host.read(path) or "").split()))

    reg.command("wc", lambda agent, arg: "counted", help="count words")
''')
reg = load(Registry(host=MemHost({'/proj/a.py': 'def a(): return 1\n'})), paths=[extdir])
reg.notes

The registered tool is an ordinary function, with the docstring the model will read, and
it works against the host it was given.

In [ ]:
reg.tools[0]('/proj/a.py'), list(reg.commands)

A file with no `setup` is loaded and left alone rather than reported as broken -- that is
how a shared helper module in the same directory should behave -- while one that fails is
reported and skipped.

In [ ]:
(extdir/'helpers.py').write_text('SHARED = 1\n')
(extdir/'broken.py').write_text('raise RuntimeError("bad import")\n')
load(Registry(), paths=[extdir]).notes

Hooking an event that does not exist is an error rather than a silent no-op, because a
misspelled hook that never fires is the hardest kind of extension bug to see.

In [ ]:
test_fail(lambda: reg.on('before_lunch', print), contains='unknown event')
EVENTS

Firing is fail-soft in the other direction: a hook that raises is recorded and the turn
continues, since an extension should not be able to end a session.

In [ ]:
reg.on('before_turn', lambda **kw: 1/0)
reg.fire('before_turn'), reg.notes[-1]

## Tool plumbing

Three things every tool shares: a clip, because the context window is the scarce resource;
a parser for the hash-verified edit commands models emit as JSON; and a probe that asks a
host whether a capability exists. `WRITE_TOOLS` names the tools that change something,
which is the line an approval policy draws.

In [ ]:
#| export
MAX_TOOL_CHARS = 6000     # per tool result; the context window is the scarce resource here
MAX_HITS = 20

# The tools that change something on disk or in the live session. Named as a set because
# that is the line an approval policy needs to draw -- see `hitl.Approvals`.
WRITE_TOOLS = frozenset({'edit_file', 'create_file', 'edit_cell', 'add_cell', 'run_python', 'memory_forget',
                         'create_skill', 'cancel_watch', 'cart_add', 'cart_remove'})


def clip(s, n=MAX_TOOL_CHARS):
    s = str(s)
    return s if len(s) <= n else s[:n] + f'\n…[{len(s)-n} more chars]'


def _cmds(commands):
    """Parse exhash commands from what a tool call can carry.

    Models emit JSON, exhash wants tuples: `[["12|a1b2|","s","old","new"]]` becomes
    `[("12|a1b2|","s","old","new")]`. Nested command tuples (`g`/`v`) recurse.
    """
    if isinstance(commands, str): commands = json.loads(commands)
    if not isinstance(commands, list): raise ValueError('commands must be a JSON list of command arrays')
    def _t(c):
        if not isinstance(c, (list, tuple)): raise ValueError(f'each command must be an array, got {type(c).__name__}')
        return tuple(_t(x) if isinstance(x, (list, tuple)) else x for x in c)
    return [_t(c) for c in commands]


def _probe(host, *calls):
    "Whether every one of `calls` is supported. A host says 'no' by raising `NotImplementedError`."
    for f in calls:
        try: f()
        except NotImplementedError: return False
        except Exception: pass
    return True

In [ ]:
clip('the whole file, all of it', 12)

Models emit JSON and exhash wants tuples, so `_cmds` converts, recursing into the nested
command arrays that `g`/`v` take.

In [ ]:
_cmds('[["12|a1b2|", "s", "old", "new"], ["30|9f3c|", "a", "appended"]]')

In [ ]:
test_fail(lambda: _cmds('{"not": "a list"}'), contains='must be a JSON list')
sorted(WRITE_TOOLS)

## Seeing the code

The first group any host gets: search the index, find code shaped like a given function,
outline a file, list the files. Every result names the exact path and how to address it,
because a model that has to guess whether something is a notebook will guess wrong.

In [ ]:
#| export
def code_tools(host):
    "Seeing the code: the index, the shapes in it, and the files it covers."

    def search_code(query: str) -> str:
        """Search the codebase and every installed package for `query`.

        Semantic when the code index is built, a literal scan otherwise. Use this before
        writing anything non-trivial: the answer is usually already in the environment.
        """
        hits = host.search(query, limit=MAX_HITS)
        if not hits: return f'no matches ({host.search_note})'
        rows = []
        for h in hits:
            target = (f'NOTEBOOK — use this exact path with notebook_cells, then view_cell/edit_cell'
                      if str(h.path).lower().endswith('.ipynb')
                      else 'FILE — use this exact path with view_file/edit_file')
            rows.append(f'{h.path}:{h.line}  {h.symbol or ""}  {h.text}\n  {target}')
        return clip(f'[{host.search_note}]\n' + '\n'.join(rows))

    def similar_code(path: str, line: int = 1) -> str:
        "Find code shaped like the function at `path`:`line` -- every place a pattern was already used."
        hits = host.peers(str(host.check(path)), int(line), limit=MAX_HITS)
        if not hits: return f'nothing similar ({host.search_note})'
        return clip('\n'.join(f'{h.path}:{h.line}  {h.symbol or ""}  {h.text}' for h in hits))

    def outline(path: str) -> str:
        "The defs and classes in one file, with line numbers."
        syms = host.symbols(str(host.check(path)))
        if not syms: return f'no symbols in {path}'
        return clip('\n'.join(f'{int(getattr(s, "score", 0))*" "}{s.line}: {s.symbol}' for s in syms))

    def list_files(pattern: str = '') -> str:
        "Files in the open folders, optionally filtered by a substring of the path."
        ps = [str(p) for p in host.walk()]
        if pattern: ps = [p for p in ps if pattern.lower() in p.lower()]
        return clip('\n'.join(ps[:400]) or 'no matching files')

    return [search_code, similar_code, outline, list_files]

In [ ]:
host = MemHost({'/proj/a.py': 'def a(): return 1\n', '/proj/b.py': 'def b(): return a() + 1\n'})
search_code, similar_code, outline, list_files = code_tools(host)
print(search_code('return'))

When nothing matches, the answer says which engine answered, so "no matches" can be told
apart from "no index".

In [ ]:
search_code('nonexistent'), list_files('b.py')

## Files

Files are read and written by hash-verified address. `view_file` returns
`lineno|hash|content` lines, and those hashes are the addresses `edit_file` takes -- so the
view is also the address book, and an edit built on a stale view fails instead of damaging
the wrong line.

In [ ]:
#| export
def file_tools(host):
    "Reading and editing files, always by hash-verified address."

    def view_file(path: str, start: int = 0, end: int = 0) -> str:
        """Read a file as `lineno|hash|content` lines. Optionally limit to lines `start`..`end`.

        Always read this way before editing: `edit_file` addresses lines by the exact
        hashes this returns, so the view is also the address book.
        """
        from exhash import lnhashview_file
        p = host.check(path)
        if not p.exists(): return f'no such file: {p}'
        return clip(str(lnhashview_file(str(p), start or None, end or None)))

    def edit_file(path: str, commands: str) -> str:
        """Edit a file with hash-verified exhash commands, and return the diff.

        `commands` is a JSON array of command arrays, each starting with an address taken
        from `view_file`, e.g.
          [["12|a1b2|", "s", "old text", "new text"],
           ["30|9f3c|", "a", "a new line appended after line 30"]]
        Every address's hash is checked immediately before it runs, so an edit built on a
        stale view fails instead of damaging the wrong line. Nothing is written unless
        every command succeeds.
        """
        from exhash import file_exhash
        p = host.check(path)
        try: cmds = _cmds(commands)
        except Exception as e: return f'could not parse commands: {agent_err(e)}'
        if not cmds: return 'no commands given'
        try: return clip(str(file_exhash(str(p), *cmds)))
        except Exception as e: return f'edit failed: {agent_err(e)}'

    def create_file(path: str, text: str = '') -> str:
        "Create (or overwrite) a whole file. For changes to an existing file prefer `edit_file`."
        try: return f'wrote {host.write(path, text)}'
        except Exception as e: return f'write failed: {agent_err(e)}'

    return [view_file, edit_file, create_file]

A real file on disk, and a host that lets the tools reach it:

In [ ]:
p = tmp/'greet.py'
p.write_text('def greet(name):\n    return "hi " + name\n')
view_file, edit_file, create_file = file_tools(NullHost([str(tmp)]))
print(view_file(str(p)))

An edit quotes an address from that view. Nothing is written unless every command
succeeds, and the diff is what comes back.

In [ ]:
line2 = view_file(str(p)).splitlines()[1]
addr = '|'.join(line2.split('|')[:2]) + '|'
print(edit_file(str(p), json.dumps([[addr, 's', '"hi "', '"hello "']])))

In [ ]:
test_eq(p.read_text(), 'def greet(name):\n    return "hello " + name\n')
addr

An address whose hash no longer matches is refused. This is the whole point of the scheme:
the model's view of line 2 is verified against the file as it is now, immediately before the
edit runs.

In [ ]:
edit_file(str(p), json.dumps([['2|0000|', 's', 'hello', 'howdy']]))

## Notebooks

The harness deliberately does not own a notebook representation: exhash addresses cells by
path and id without one, so only the two operations that genuinely need to know what a
notebook *is* are delegated to the host.

In [ ]:
#| export
def notebook_tools(host):
    "Notebooks, addressed by cell id rather than by line."

    def notebook_cells(path: str) -> str:
        "List a notebook's cells: id, type, and first line. Cell ids are what `edit_cell` addresses."
        try: rows = host.nb_cells(str(host.check(path)))
        except NotImplementedError: raise
        except Exception as e: return f'could not read notebook: {agent_err(e)}'
        return clip('\n'.join(f'{i}  {t:8} {(s or "").strip().splitlines()[0][:100] if (s or "").strip() else ""}'
                              for i, t, s in rows) or '(empty notebook)')

    def view_cell(path: str, cell_id: str) -> str:
        "Read one notebook cell as `lineno|hash|content` lines, ready to address with `edit_cell`."
        from exhash import lnhashview_cell
        try: return clip(str(lnhashview_cell(str(host.check(path)), cell_id)))
        except Exception as e: return f'could not read cell: {agent_err(e)}'

    def edit_cell(path: str, cell_id: str, commands: str) -> str:
        "Edit one notebook cell's source with exhash commands from `view_cell`. Same format as `edit_file`."
        from exhash import cell_exhash
        try: cmds = _cmds(commands)
        except Exception as e: return f'could not parse commands: {agent_err(e)}'
        try: return clip(str(cell_exhash(str(host.check(path)), cell_id, *cmds)))
        except Exception as e: return f'edit failed: {agent_err(e)}'

    def add_cell(path: str, source: str, index: int = -1, cell_type: str = 'code') -> str:
        "Insert a new cell into a notebook at `index` (-1 appends). Creates the notebook if needed."
        try: return f'added cell {host.nb_add_cell(str(host.check(path)), source, int(index), cell_type)} to {path}'
        except NotImplementedError: raise
        except Exception as e: return f'could not add cell: {agent_err(e)}'

    return [notebook_cells, view_cell, edit_cell, add_cell]

A host that cannot do notebooks raises, and these tools are never built -- which is what
the next section's probe is for.

In [ ]:
notebook_cells, view_cell, edit_cell, add_cell = notebook_tools(host)
with expect_fail(NotImplementedError): notebook_cells('/proj/x.ipynb')
[t.__name__ for t in notebook_tools(host)]

## The web and remembered research

Two groups, and the split matters: the web tools go out now, while the memory tools recall
pages read earlier as whole document sections. A question that was researched last week
should cost a memory search rather than another crawl.

In [ ]:
#| export
def web_tools(host):
    "The web, for the questions whose answer depends on current documentation."

    def web_search(query: str) -> str:
        "Search the web. Returns titles and urls; follow up with `read_url` on the useful ones."
        docs = host.web_search(query, n=MAX_HITS)
        if not docs: return f'no results ({host.research_note})'
        return clip('\n'.join(f'{d.title}\n  {d.url}' for d in docs))

    def read_url(url: str, remember: bool = True) -> str:
        """Read one web page (or GitHub file, arxiv paper) as markdown.

        It enters durable research memory by default. Pass `remember=False` for sensitive,
        obviously irrelevant, or exploratory results that should remain ephemeral.
        """
        d = host.read_url(url, remember=remember)
        return clip(d.text if d else f'could not read {url} ({host.research_note})')

    def research(query: str) -> str:
        "Search the web and read the top results into one cited digest. Slower than `web_search`; use for depth."
        return clip(host.research(query) or f'nothing found ({host.research_note})')

    return [web_search, read_url, research]

In [ ]:
#| export
def memory_tools(host):
    "Durable pages and research recalled as document sections rather than flat snippets."

    def memory_search(query: str, limit: int = 8) -> str:
        """Search pages remembered from earlier reads and research.

        Returns whole operative sections with breadcrumbs plus related semantic paths. Use
        this before searching the live web when the question may have been researched before.
        """
        try: return clip(json.dumps(host.memory_search(query, int(limit)), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return f'memory search failed: {agent_err(e)}'

    def memory_tree(document: str = '') -> str:
        """Browse remembered document headings without embedding a query.

        `document` may be a title substring or stable document id. Leave it empty to list
        all remembered roots, then call again with the relevant document.
        """
        try: return clip(json.dumps(host.memory_tree(document), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return f'memory tree failed: {agent_err(e)}'

    def memory_read(node_id: str) -> str:
        "Read one whole remembered section by the node id returned by memory_search/tree."
        try: return clip(json.dumps(host.memory_read(node_id), default=str), MAX_TOOL_CHARS * 3)
        except Exception as e: return f'memory read failed: {agent_err(e)}'

    def memory_topics(limit: int = 12) -> str:
        "Map remembered material into labelled semantic clusters and representative members."
        try: return clip(json.dumps(host.memory_topics(int(limit)), default=str), MAX_TOOL_CHARS * 2)
        except Exception as e: return f'memory topics failed: {agent_err(e)}'

    def memory_forget(doc_id: str) -> str:
        """Purge one bad, sensitive, stale or irrelevant remembered document by id.

        This removes its tree, chunks and ANN entries. Use only when the user requests it;
        do not silently curate their memory.
        """
        try: return 'forgot document' if host.memory_forget(doc_id) else 'document was not forgotten'
        except Exception as e: return f'memory purge failed: {agent_err(e)}'

    return [memory_search, memory_tree, memory_read, memory_topics, memory_forget]

In [ ]:
[t.__name__ for t in web_tools(host)], [t.__name__ for t in memory_tools(host)]

`NullHost.web_search` returns nothing rather than raising, so the tool exists and reports
honestly -- the distinction the note carries.

In [ ]:
web_search, read_url, research = web_tools(host)
web_search('nbdev v3 export'), memory_tools(host)[0]('nbdev')

## The live session

The user's kernel, and the terminal they are looking at. `run_python` lands in their
namespace and is a write tool; `inspect_python` cannot change anything and is not, which is
why the briefing tells the model to reach for it first. `read_terminal` is read-only by
construction: it shows what the user ran and cannot run anything.

`memory_tools` recalls what was read. `watch_tools` is the other direction: what the agent
arranged to read later. The two share a store deliberately -- a reminder that fires becomes an
ordinary note, so "what am I supposed to be doing" and "what do I know" are one query, and
neither needs a notification channel the harness does not have.

In [ ]:
#| export
def watch_tools(host):
    "Standing interests: what to put back on the desk later, and what has come due now."

    def remember(text: str, title: str = '', tags: str = '') -> str:
        """Write a conclusion into durable memory so a later session finds it.

        For what you worked out, not for what you read -- `read_url` already files pages.
        `tags` is a comma-separated list.
        """
        try:
            d = host.remember(text, title=title or None,
                              tags=[t.strip() for t in tags.split(',') if t.strip()])
            return f"remembered {d.get('title')!r} as {d.get('doc_id')}"
        except Exception as e: return f'could not remember: {agent_err(e)}'

    def set_reminder(text: str, every: str = '1w', note: str = '') -> str:
        """Come back to `text` every `every` ('30m', '6h', '1d', '1w').

        The reminder files itself into memory when it comes due, so it surfaces in
        `memory_search` and in `poll_watches` rather than needing a notification channel.
        """
        try:
            w = host.watch(text, action='remind', every=every, note=note or None)
            return f"reminder {w['id']} set, every {every}"
        except Exception as e: return f'could not set reminder: {agent_err(e)}'

    def watch_url(url: str, every: str = '1d', note: str = '') -> str:
        "Re-read `url` every `every` and file each version in memory, so changes are visible over time."
        try:
            w = host.watch(url, action='url', every=every, note=note or None)
            return f"watching {url} as {w['id']}, every {every}"
        except Exception as e: return f'could not watch: {agent_err(e)}'

    def list_watches(due_only: bool = False) -> str:
        "Every standing watch and reminder, soonest first. `due_only` shows just what has come due."
        try:
            ws = host.watches(due_only=bool(due_only))
            if not ws: return 'nothing is being watched'
            return clip('\n'.join(
                f"{w['id']}  {w['action']:8} every {int(w['every'])}s  runs={w['runs']}"
                f"  {w.get('last_status') or 'never run'}  {str(w['target'])[:80]}" for w in ws))
        except Exception as e: return f'could not list watches: {agent_err(e)}'

    def cancel_watch(watch_id: str) -> str:
        "Delete one watch by id. Only when the user asks; do not silently curate their reminders."
        try:
            host.unwatch(watch_id)
            return f'cancelled {watch_id}'
        except Exception as e: return f'could not cancel: {agent_err(e)}'

    def poll_watches() -> str:
        """Run every watch that has come due, and report what fired.

        Call this when the user asks what is outstanding, or at the start of a session.
        Anything that fired is now in memory: follow up with `memory_search`.
        """
        try:
            r = host.poll()
            if not r.get('ran'): return f"nothing due ({r.get('checked', 0)} watched)"
            lines = [f"{x['status']:7} {x['action']:8} {str(x['target'])[:90]}" for x in r['results']]
            return clip(f"{r['ran']} of {r['checked']} fired\n" + '\n'.join(lines))
        except Exception as e: return f'poll failed: {agent_err(e)}'

    return [remember, set_reminder, watch_url, list_watches, cancel_watch, poll_watches]

In [ ]:
#| export
def session_tools(host):
    "The live kernel the user is working in, and the terminal they are looking at."

    def list_vars() -> str:
        "List the variables visible in the user's live session: name, type, and a short value."
        return clip(host.list_vars() or '(empty session)')

    def run_python(code: str) -> str:
        """Run Python in the user's live kernel namespace.

        Read any variable freely; bind results to NEW names so they survive to the next
        call. Mutating or deleting the user's variables is refused -- rebind instead
        (`df2 = df.drop(...)`). Call `list_vars` first if you do not know what is there.
        """
        try: return clip(host.run_python(code))
        except NotImplementedError: raise
        except Exception as e: return f'run failed: {agent_err(e)}'

    def scale_numeric(source: str = 'df', output: str = 'df_norm') -> str:
        """Min-max scale a DataFrame's numeric columns to 0..1 in a new live variable.

        Use this instead of composing pandas code when the user asks to scale an existing
        DataFrame. Non-numeric columns are copied unchanged; constant numeric columns are
        set to zero. `source` and `output` must be top-level variable names.
        """
        if not re.fullmatch(r'[A-Za-z_]\w*', source or ''): return 'source must be a variable name'
        if not re.fullmatch(r'[A-Za-z_]\w*', output or ''): return 'output must be a variable name'
        code = (f"{output} = {source}.copy()\n"
                f"_lee_num = {source}.select_dtypes(include='number').columns\n"
                f"_lee_min = {source}[_lee_num].min()\n"
                f"_lee_span = ({source}[_lee_num].max() - _lee_min).replace(0, 1)\n"
                f"{output}[_lee_num] = ({source}[_lee_num] - _lee_min) / _lee_span\n"
                f"{output}.head()")
        try: return clip(host.run_python(code))
        except NotImplementedError: raise
        except Exception as e: return f'scale failed: {agent_err(e)}'

    def inspect_python(code: str, scope: str = 'isolated') -> str:
        """Look at the user's live variables by running Python that cannot change them.

        Two scopes. Both leave the user's variables exactly as they were; they differ in
        how much Python you get, so pick by what the question needs:

        - `scope='isolated'` (default) runs in an allowlist sandbox on a copy. Attribute
          reads and builtins work — `df.shape`, `len(df)`, `type(x).__name__` — and most
          library method calls are refused. Costs nothing to be wrong about.
        - `scope='overlay'` runs the real interpreter against the real namespace. Library
          calls work: `list(df.columns)`, `df.head(3).to_dict()`, `model.summary()`. Names
          you bind persist into your own layer for later calls. You still cannot delete,
          rebind or mutate anything the user made — that is refused, with an explanation.

        Start isolated; move to overlay when the sandbox refuses something you need. Neither
        needs approval, and both run while one of the user's cells is still going. For work
        that must land in the *user's* namespace, use `run_python` instead.
        """
        try: return clip(host.inspect_python(code, scope=scope))
        except NotImplementedError: raise
        except Exception as e: return f'inspection failed: {agent_err(e)}'

    def read_terminal(lines: int = 200) -> str:
        """Read what the IDE's terminal has printed -- a failing build, a stack trace, a test run.

        This is *read only*: it shows what the user ran, and cannot run anything. Use it
        when they mention an error they are looking at rather than asking them to paste it.
        """
        return clip(host.terminal_text(int(lines)) or 'the terminal has printed nothing yet')

    return [list_vars, run_python, scale_numeric, inspect_python, read_terminal]

In [ ]:
list_vars, run_python, scale_numeric, inspect_python, read_terminal = session_tools(host)
run_python('df2 = df.dropna()'), host.ran

`scale_numeric` is a tool rather than advice because the model composing this pandas by
hand gets the constant-column case wrong. Variable names are validated before anything is
composed.

In [ ]:
scale_numeric('df', 'df; import os'), host.ran[-1].splitlines()[0]

## Skills as tools

`read_skill` is what makes the index in the system prompt affordable: names and
descriptions go out with every turn, bodies only when asked for. `create_skill` writes a
project-local `SKILL.md`, and never overwrites one.

In [ ]:
#| export
def skill_tools(host, get_skills):
    "Reading discovered skills and creating project-local Agent Skills."

    def read_skill(name: str) -> str:
        """Read one skill in full: how to use a tool or a library that is already installed here.

        The skill list in your briefing gives names and one-line descriptions. Read the
        matching one *before* doing the work it describes, not after it has gone wrong.
        """
        ss = get_skills()
        s = find(ss, name)
        if s is None:
            return f'no skill matching {name!r}. Available: ' + ', '.join(x.name for x in ss)
        return clip(f'<skill name="{s.name}" from="{s.where}">\n{s.text()}\n</skill>', MAX_TOOL_CHARS * 3)

    def create_skill(name: str, description: str, instructions: str) -> str:
        """Create a reusable project skill at `.agents/skills/NAME/SKILL.md`.

        Use this only when the user asks to preserve repeatable project know-how as a
        skill, not for ordinary task notes. `name` must be lowercase kebab-case;
        `description` says when it applies; `instructions` is the complete Markdown body.
        Existing skills are never overwritten. Run `/reload` after creation to make the
        current agent advertise it immediately.
        """
        from pathlib import Path
        name = str(name or '').strip()
        if not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*', name):
            return 'skill name must be lowercase kebab-case (for example, notebook-tests)'
        if not str(description or '').strip(): return 'skill description is required'
        if not str(instructions or '').strip(): return 'skill instructions are required'
        roots = list(host.roots or ())
        if not roots: return 'open a project folder before creating a skill'
        target = Path(host.check(Path(roots[0])/'.agents'/'skills'/name/'SKILL.md'))
        exists = target.exists()
        if not exists:
            try: exists = host.read(str(target)) is not None
            except Exception: pass
        if exists: return f'refusing to overwrite existing skill: {target}'
        title = json.dumps(name, ensure_ascii=False)
        desc = json.dumps(' '.join(str(description).split()), ensure_ascii=False)
        text = f'---\nname: {title}\ndescription: {desc}\n---\n\n{str(instructions).strip()}\n'
        try: host.write(str(target), text)
        except Exception as e: return f'could not create skill: {agent_err(e)}'
        return f'created {target}; run /reload to load it into the current agent'

    return [read_skill, create_skill]

In [ ]:
read_skill, create_skill = skill_tools(host, lambda: skills)
print(read_skill('notebook-tests')[:120])

An unresolvable name answers with the list, rather than with nothing.

In [ ]:
read_skill('nope')[:80]

A created skill lands under the first open folder, in the layout `discover` reads, and a
name that is not kebab-case is refused before anything is written.

In [ ]:
create_skill('Notebook Tests', 'when testing', 'body'), create_skill('nb-tests', 'when testing notebooks', 'Use `test_eq`.')

In [ ]:
test_eq(create_skill('nb-tests', 'again', 'body').startswith('refusing to overwrite'), True)
sorted(host.files)

## Assembling the tool list

`tools_for` probes each group with a harmless call and drops the whole group when the host
does not implement it. Whole groups rather than individual tools, because the groups are the
real units of capability: a host with no notebook representation cannot support any of the
four notebook tools.

In [ ]:
#| export
def tools_for(host, get_skills=None, extra=()):
    """Every tool this host can actually support, plus whatever extensions registered.

    Each group is probed with a harmless call and dropped whole if the host does not
    implement it. Whole groups rather than individual tools because the groups are the
    real units of capability: a host with no notebook representation cannot support any of
    the four notebook tools, and one with no kernel cannot support any of the session ones.
    """
    tools = []
    tools += code_tools(host)
    tools += file_tools(host)
    if _probe(host, lambda: host.nb_cells('.')): tools += notebook_tools(host)
    if _probe(host, lambda: host.web_search('', n=1)): tools += web_tools(host)
    if _probe(host, lambda: host.memory_tree('')): tools += memory_tools(host)
    if _probe(host, lambda: host.watches()): tools += watch_tools(host)
    if _probe(host, lambda: host.list_vars(), lambda: host.terminal_text(1)): tools += session_tools(host)
    if get_skills is not None: tools += skill_tools(host, get_skills)
    tools += list(extra or ())
    return tools

A `MemHost` supports code, files and (emptily) the web, but has no notebooks, no memory and
no variable listing -- so it receives only the groups it can actually run:

In [ ]:
ts = tools_for(host)
[t.__name__ for t in ts]

In [ ]:
test_eq(_probe(host, lambda: host.nb_cells('.')), False)
test_eq(_probe(host, lambda: host.search('x')), True)
len(ts), len(tools_for(host, get_skills=lambda: skills))

A `LocalHost` over a real folder supports notebooks and a live session as well, so the same
call builds a much longer list -- and remembered research is still absent from both, because
neither has a memory index to search.

In [ ]:
[t.__name__ for t in tools_for(local)]

## Sub-agents

Delegation is a context strategy, not a speed one. A broad question that takes twenty tool
calls to answer costs the caller one question and one answer, because the sub-agent's
working is discarded with its conversation. A sub-agent gets read-only tools, and cannot
delegate further: recursion here is a fan-out tree whose width nobody chose.

In [ ]:
#| export
SUB_MAX_STEPS = 12

SUB_SP = """You are a research sub-agent inside a Python IDE. Another agent has delegated one \
question to you because answering it takes many tool calls and the answer is short.

- Answer exactly the question asked. Nothing else.
- Use your tools as much as you need; nobody is paying attention to how many calls it takes.
- Report what you found, with file paths and line numbers, not what you infer or expect.
- If the answer is that there is nothing, say so plainly. A confident wrong answer is far \
worse than "no matches, and here is what I searched for".
- You cannot edit anything. If the answer implies a change, describe the change and stop.
- `inspect_python` answers questions about the user's live variables without changing them. \
Its default scope is a sandbox that refuses most library calls; pass `scope='overlay'` to \
get the real interpreter. Use it rather than guessing at what is in memory."""


# A sub-agent does not get to spawn sub-agents. Not a safety rule so much as an economic
# one: recursion here is a fan-out tree whose width nobody chose, and the second level
# never has enough context to ask a good question anyway.
NO_SUB = frozenset({'delegate_search', 'delegate_parallel'})

In [ ]:
#| export
def read_only(tools, max_calls=None):
    "The read-only tools a sub-agent may have, optionally behind a hard per-task call budget."
    allowed = [t for t in tools if getattr(t, '__name__', '') not in (WRITE_TOOLS | NO_SUB)]
    if max_calls is None: return allowed
    state, lock = {'n': 0}, threading.Lock()

    def guarded(f):
        @functools.wraps(f)
        def call(*args, **kw):
            with lock:
                state['n'] += 1
                over = state['n'] > max_calls
            if over:
                return ('Sub-agent tool budget exhausted. Stop calling tools and return the '
                        'best evidence-backed answer now.')
            return f(*args, **kw)
        return call
    return [guarded(t) for t in allowed]

Every write tool and both delegation tools are filtered out, whatever else the host
offered:

In [ ]:
[t.__name__ for t in read_only(ts)]

In [ ]:
test_eq(set(t.__name__ for t in read_only(ts)) & WRITE_TOOLS, set())
sorted(NO_SUB)

With a budget, the tools themselves stop the loop. Local engines own their internal tool
loop, so the wrapper is the one hard stop that works on every backend.

In [ ]:
budgeted = read_only(ts, max_calls=1)
search = next(t for t in budgeted if t.__name__ == 'search_code')
search('return'), search('return')

`delegate` runs one question in a throwaway conversation on the same engine, and closes it
in a `finally` -- a sub-agent whose context leaks back into the session is just a slower way
of doing the work inline.

In [ ]:
#| export
def delegate(backend, question, tools=(), sp=SUB_SP, max_steps=SUB_MAX_STEPS):
    """Ask `question` in a throwaway conversation on `backend`'s engine. Returns the answer text.

    The conversation is closed in a `finally` because the whole benefit is that it does not
    outlive the question -- a sub-agent whose context leaks back into the session is just a
    slower way of doing the work inline.
    """
    sub = None
    try:
        # Native local engines own their internal tool loop, so the tool wrappers are the
        # backend-independent hard stop. Allow several parallel calls per logical round.
        sub = backend.spawn(sp=sp, tools=read_only(tools, max_calls=max_steps * 4))
        if hasattr(sub, 'max_steps'): sub.max_steps = max_steps
        return sub.send(question)
    except Exception as e:
        return f'delegation failed: {agent_err(e)}'
    finally:
        if sub is not None:
            try: sub.close()
            except Exception: pass

In [ ]:
#| export
def delegate_many(backend, questions, tools=(), sp=SUB_SP, max_steps=SUB_MAX_STEPS, n_workers=4):
    """Ask several questions at once. Returns answers in the order the questions were given.

    Whether this is genuinely parallel depends on what is underneath, and it is worth being
    exact rather than optimistic:

    - **Generation** overlaps on a cloud backend, where each sub-agent is its own HTTP
      request. On a local engine it does not -- litert holds one conversation at a time --
      so local fan-out is run one after another rather than racing for the same engine and
      finding out what happens.
    - **Tool work** overlaps either way, and is usually the bulk of it: three sub-agents
      each doing six searches is eighteen searches, and they do not wait for each other.
      Under a concurrent kernel (`Host.kernel_kind == 'ipymini'`) that includes
      `inspect_python`, which is the case that used to be hopeless -- an inspection queued
      behind the user's running cell, and then behind the other two sub-agents' inspections.

    The point of the whole thing is context, not speed. Three questions answered in
    parallel cost the caller three short answers instead of sixty tool results.
    """
    qs = list(questions)
    if not qs: return []
    if len(qs) == 1 or getattr(backend.spec, 'local', False) or n_workers < 2:
        return [delegate(backend, q, tools, sp, max_steps) for q in qs]
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(min(n_workers, len(qs))) as ex:
        return list(ex.map(lambda q: delegate(backend, q, tools, sp, max_steps), qs))

In [ ]:
be = FakeBackend(replies=['the caller never sees this'])
delegate(be, 'which files import fastllm?', tools=ts)

The spawned conversation is separate, and gone by the time the answer is returned.

In [ ]:
test_eq(len(be.spawned), 1)
be.spawned[0].hist

`delegate_many` keeps the answers in the order the questions were asked. On a local model
it runs them one after another on purpose: litert holds one conversation at a time, so
fanning out would mean racing for the same engine to find out what happens.

In [ ]:
delegate_many(be, ['what imports fastllm?', 'where is compaction triggered?'], tools=ts)

The tools themselves take callables rather than a backend, so a model switch mid-session is
picked up -- the tool the model is holding must not be pinned to whichever backend happened
to be current when the list was built.

In [ ]:
#| export
def subagent_tools(get_backend, get_tools):
    """The `delegate` tool, bound to whatever backend routing says sub-agents run on.

    Both arguments are callables so a model switch mid-session is picked up: the tool the
    model is holding must not be pinned to the backend that happened to be current when
    the tool list was built.
    """

    def delegate_search(question: str) -> str:
        """Hand a broad search question to a sub-agent and get back only its conclusion.

        Use this when answering would take many `search_code` / `view_file` / `read_url` /
        `inspect_python` calls whose results you do not need to keep -- "where else do we
        do X", "which files import Y", "what shape is everything in this namespace". The
        sub-agent has your read-only tools and none of your write tools, and its working is
        discarded, so the cost to your context is one question and one answer.

        Ask one self-contained question. The sub-agent cannot see this conversation.
        """
        b = get_backend()
        if b is None: return 'no model is available to delegate to'
        return clip(delegate(b, question, get_tools()), MAX_TOOL_CHARS)

    def delegate_parallel(questions: str) -> str:
        """Hand several independent questions to sub-agents at once, and get back every answer.

        `questions` is a JSON array of strings, e.g.
          ["which files import fastllm?", "where is compaction triggered?", "what is df's shape?"]

        Use it when you have two or more questions that do not depend on each other. They
        run concurrently, each in its own throwaway conversation with your read-only tools,
        so three questions cost you three short answers rather than the sixty tool results
        it would take to answer them yourself.

        Every question must be self-contained: a sub-agent cannot see this conversation or
        the other questions.
        """
        b = get_backend()
        if b is None: return 'no model is available to delegate to'
        try:
            qs = json.loads(questions) if isinstance(questions, str) else list(questions)
            if not isinstance(qs, list) or not all(isinstance(q, str) for q in qs):
                raise ValueError('expected a JSON array of strings')
        except Exception as e:
            return f'could not parse questions: {agent_err(e)}'
        if not qs: return 'no questions given'
        answers = delegate_many(b, qs, get_tools())
        return clip('\n\n'.join(f'### {q}\n{a}' for q, a in zip(qs, answers)), MAX_TOOL_CHARS * 2)

    return [delegate_search, delegate_parallel]

In [ ]:
delegate_search, delegate_parallel = subagent_tools(lambda: be, lambda: ts)
[t.__name__ for t in subagent_tools(lambda: be, lambda: ts)]

With no model available it says so, rather than raising into the turn.

In [ ]:
test_eq(subagent_tools(lambda: None, lambda: ts)[0]('anything'), 'no model is available to delegate to')
delegate_parallel('["what imports fastllm?"]')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()